# 🕵️‍♀️ Scientific Image Forgery Detection: DINOv2 High-Res
### **Score: 0.332 (Public LB)**

This notebook implements a robust semantic segmentation pipeline for detecting copy-move and splicing forgeries in scientific imagery.

### **Key Features:**
* **Backbone:** `DINOv2 (Base)` - Frozen Vision Transformer features.
* **Resolution:** **4500px** - Upgraded input size to preserve microscopic artifacts.
* **Inference:** Sparse Sliding Window (`Stride=400`) + TTA.
* **Post-Processing:** Strict probability thresholding (`0.19`) with adaptive cleaning.

### **Architecture:**
The model uses a custom "Tiny Decoder" that upsamples DINOv2's `14x14` patch embeddings into a high-fidelity segmentation mask.

## 1. Configuration
We prioritize **High Fidelity** inputs (4500px) while maintaining speed using a sparse stride (400).
* **MAX_IMG_SIZE = 4500:** Prevents downscaling artifacts on large scientific figures.
* **Threshold = 0.19:** A proven "hard floor" that rejects background noise while catching faint forgeries.

In [ ]:
import os
import cv2
import json
import math
import random
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModel
from scipy.ndimage import binary_fill_holes
import time

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    
    # 🚀 SPEED OPTIMIZATION
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True 

seed_everything(42)

class CONFIG:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Paths
    BASE_DIR  = "/kaggle/input/recodai-luc-scientific-image-forgery-detection"
    TEST_DIR  = f"{BASE_DIR}/test_images"
    DINO_PATH = "/kaggle/input/dinov2/pytorch/base/1"
    MODEL_LOC = '/kaggle/input/cnndinov2-pbd/CNNDINOv2-U52/CNNDINOv2-U52/model_seg_final.pt'
    MAX_IMG_SIZE = 3000     
    window_size = 518       
    stride = 300            
    use_tta = True          
    batch_size = 32         
    min_mean_conf = 0.19    
    alpha_grad = 0.50       
    min_pixel_size = 50     
    TIME_LIMIT_HOURS = 8.5 

## 2. Model Architecture
A lightweight CNN decoder is attached to the DINOv2 backbone. It progressively upsamples the feature maps to recover spatial resolution.

In [ ]:
class DinoTinyDecoder(nn.Module):
    def __init__(self, in_ch=768, out_ch=1):
        super().__init__()
        self.block1 = nn.Sequential(nn.Conv2d(in_ch, 384, 3, 1, 1), nn.ReLU(True), nn.Dropout2d(0.1))
        self.block2 = nn.Sequential(nn.Conv2d(384, 192, 3, 1, 1), nn.ReLU(True), nn.Dropout2d(0.1))
        self.block3 = nn.Sequential(nn.Conv2d(192, 96, 3, 1, 1), nn.ReLU(True))
        self.conv_out = nn.Conv2d(96, out_ch, 1)
    
    def forward(self, f, target_size):
        x = F.interpolate(self.block1(f), size=(74, 74), mode='bilinear', align_corners=False)
        x = F.interpolate(self.block2(x), size=(148, 148), mode='bilinear', align_corners=False)
        x = F.interpolate(self.block3(x), size=(296, 296), mode='bilinear', align_corners=False)
        x = self.conv_out(x)
        return F.interpolate(x, size=target_size, mode='bilinear', align_corners=False)

class DinoSegmenter(nn.Module):
    def __init__(self, encoder, processor):
        super().__init__()
        self.encoder, self.processor = encoder, processor
        self.seg_head = DinoTinyDecoder(768, 1)
        
    def forward_features(self, x):
        imgs = (x * 255).clamp(0, 255).byte().permute(0, 2, 3, 1).cpu().numpy()
        inputs = self.processor(images=list(imgs), return_tensors="pt").to(x.device)
        feats = self.encoder(**inputs).last_hidden_state
        B, N, C = feats.shape
        fmap = feats[:, 1:, :].permute(0, 2, 1)
        s = int(math.sqrt(N - 1))
        fmap = fmap.reshape(B, C, s, s)
        return fmap
        
    def forward_seg(self, x):
        fmap = self.forward_features(x)
        return self.seg_head(fmap, (CONFIG.window_size, CONFIG.window_size))

## 3. Sliding Window & TTA
To handle 4K+ resolution images without OOM errors, we use a sliding window approach with **Edge Padding**.
* **TTA:** Standard 3-Way Test Time Augmentation (Flip H, Flip V).
* **Fusion:** We combine Global context (resized image) with Local details (crops) using a 0.4/0.6 weighted average.

In [ ]:
def load_model_safely():
    try:
        processor = AutoImageProcessor.from_pretrained(CONFIG.DINO_PATH, local_files_only=True, use_fast=False)
        encoder = AutoModel.from_pretrained(CONFIG.DINO_PATH, local_files_only=True).eval().to(CONFIG.device)
        model = DinoSegmenter(encoder, processor).to(CONFIG.device)
        if os.path.exists(CONFIG.MODEL_LOC):
            model.load_state_dict(torch.load(CONFIG.MODEL_LOC, map_location=CONFIG.device))
            print(f"✅ Loaded weights: {CONFIG.MODEL_LOC}")
        model.eval()
        return model
    except Exception as e:
        print(f"Fatal Error: {e}")
        return None

model = load_model_safely()

## 4. Adaptive Mask Generation
The probability map is refined using:
1.  **Sobel Edge Boosting:** Enhances boundaries of spliced regions.
2.  **Morphological Cleaning:** A `7x7` Closing operation connects fragmented forgery blobs.
3.  **Safety Filter:** Blobs smaller than 100px are removed to prevent False Positives.


In [ ]:
@torch.no_grad()
def predict_batch(images_list, model):
    batch_np = np.stack(images_list)
    x = torch.from_numpy(batch_np).float().permute(0,3,1,2) / 255.0
    x = x.to(CONFIG.device)
    
    # 1. Original
    pred = torch.sigmoid(model.forward_seg(x))
    
    if CONFIG.use_tta:
        # 2. Flip H
        pred_h = torch.flip(torch.sigmoid(model.forward_seg(torch.flip(x, [3]))), [3])
        # 3. Flip V
        pred_v = torch.flip(torch.sigmoid(model.forward_seg(torch.flip(x, [2]))), [2])
        # 4. Flip HV (Rotate 180) - Extra stability
        pred_hv = torch.flip(torch.sigmoid(model.forward_seg(torch.flip(x, [2, 3]))), [2, 3])
        
        # 4-Way Average
        pred = (pred + pred_h + pred_v + pred_hv) / 4.0
        
    return pred.cpu().numpy()[:, 0, :, :]

def sliding_window_inference(pil_img, model):
    w_orig, h_orig = pil_img.size
    
    if w_orig <= CONFIG.window_size and h_orig <= CONFIG.window_size:
        pad_w = max(0, CONFIG.window_size - w_orig)
        pad_h = max(0, CONFIG.window_size - h_orig)
        img_np = np.array(pil_img)
        img_padded = np.pad(img_np, ((0, pad_h), (0, pad_w), (0, 0)), mode='edge')
        pred = predict_batch([img_padded], model)[0]
        return pred[:h_orig, :w_orig]

    full_prob = np.zeros((h_orig, w_orig), dtype=np.float32)
    count_map = np.zeros((h_orig, w_orig), dtype=np.float32)
    img_np = np.array(pil_img)
    
    crops, coords = [], []
    for y in range(0, h_orig, CONFIG.stride):
        for x in range(0, w_orig, CONFIG.stride):
            y_start = min(y, h_orig - CONFIG.window_size)
            x_start = min(x, w_orig - CONFIG.window_size)
            y_start = max(0, y_start); x_start = max(0, x_start)
            
            crop = img_np[y_start:y_start+CONFIG.window_size, x_start:x_start+CONFIG.window_size]
            if crop.shape[0] != CONFIG.window_size or crop.shape[1] != CONFIG.window_size:
                 pad_h = CONFIG.window_size - crop.shape[0]
                 pad_w = CONFIG.window_size - crop.shape[1]
                 crop = np.pad(crop, ((0, pad_h), (0, pad_w), (0, 0)), mode='edge')
            crops.append(crop)
            coords.append((y_start, x_start))
            
    for i in range(0, len(crops), CONFIG.batch_size):
        batch_crops = crops[i:i+CONFIG.batch_size]
        batch_coords = coords[i:i+CONFIG.batch_size]
        batch_preds = predict_batch(batch_crops, model)
        if len(batch_crops) == 1 and len(batch_preds.shape) == 3:
             batch_preds = np.expand_dims(batch_preds, axis=0)
        
        for pred, (y, x) in zip(batch_preds, batch_coords):
            h_actual = min(CONFIG.window_size, h_orig - y)
            w_actual = min(CONFIG.window_size, w_orig - x)
            full_prob[y:y+h_actual, x:x+w_actual] += pred[:h_actual, :w_actual]
            count_map[y:y+h_actual, x:x+w_actual] += 1
            
    return full_prob / (count_map + 1e-6)

def enhanced_adaptive_mask(prob, alpha_grad=0.50):
    gx = cv2.Sobel(prob, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(prob, cv2.CV_32F, 0, 1, ksize=3)
    grad = np.sqrt(gx**2 + gy**2)
    grad_max = grad.max()
    if grad_max > 0: grad /= grad_max
    
    enhanced = (1 - alpha_grad) * prob + alpha_grad * grad
    enhanced = cv2.GaussianBlur(enhanced, (3,3), 0)
    
    # Hard Floor Threshold (0.19)
    mask = (enhanced > CONFIG.min_mean_conf).astype(np.uint8)
    
    # 7x7 Close (Proven Best)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((7,7), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
    
    return mask, enhanced

def pipeline_fusion(pil_image, model):
    w_orig, h_orig = pil_image.size
    
    # 1. Global (Lanczos)
    img_global = pil_image.resize((CONFIG.window_size, CONFIG.window_size), Image.LANCZOS)
    prob_global_small = predict_batch([np.array(img_global)], model)[0]
    prob_global = cv2.resize(prob_global_small, (w_orig, h_orig), interpolation=cv2.INTER_LINEAR)
    
    # 2. Local
    if max(w_orig, h_orig) > CONFIG.window_size * 1.2:
        prob_detail = sliding_window_inference(pil_image, model)
        # Fixed 0.4/0.6 Fusion (Proven Stability)
        final_prob = (0.4 * prob_global) + (0.6 * prob_detail)
    else:
        final_prob = prob_global
        
    # 3. Post-Process
    mask, enhanced_prob = enhanced_adaptive_mask(final_prob, alpha_grad=CONFIG.alpha_grad)
    
    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    final_mask = np.zeros_like(mask)
    
    for i in range(1, num):
        area = stats[i, cv2.CC_STAT_AREA]
        if area < CONFIG.min_pixel_size: continue
        
        blob_mean = enhanced_prob[labels == i].mean()
        if blob_mean > CONFIG.min_mean_conf:
            final_mask[labels == i] = 1
            
    final_mask = binary_fill_holes(final_mask).astype(np.uint8)
    
    if final_mask.sum() == 0:
        return "authentic", final_mask
    return "forged", final_mask

def rle_encode(mask):
    pixels = mask.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    if len(runs) == 0: return "authentic"
    return json.dumps([int(x) for x in runs])

## 5. Main Loop
Processing test images with a time limit safety check.

In [ ]:
def run_inference():
    rows = []
    test_files = sorted(os.listdir(CONFIG.TEST_DIR))
    total_files = len(test_files)
    
    START_TIME = time.time()
    LIMIT_SECONDS = CONFIG.TIME_LIMIT_HOURS * 3600
    
    print(f"🚀 Running Forecasting Robust: 3000px + Stride 300 (Dense) + 4-Way TTA")
    
    for i, f in tqdm(enumerate(test_files), total=total_files):
        if time.time() - START_TIME > LIMIT_SECONDS:
            print(f"⚠️ Limit reached (8.5h). Filling remaining {total_files - i}.")
            for r_f in test_files[i:]:
                 rows.append({"case_id": Path(r_f).stem, "annotation": "authentic"})
            break
            
        try:
            pil = Image.open(Path(CONFIG.TEST_DIR)/f).convert("RGB")
            
            w, h = pil.size
            scale = 1.0
            if max(w, h) > CONFIG.MAX_IMG_SIZE:
                scale = CONFIG.MAX_IMG_SIZE / max(w, h)
                pil = pil.resize((int(w*scale), int(h*scale)), Image.BILINEAR)
            
            label, mask = pipeline_fusion(pil, model)
            
            if scale != 1.0:
                mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
            
            if label == "authentic":
                annot = "authentic"
            else:
                annot = rle_encode(mask)
        except Exception as e:
            print(f"Error {f}: {e}")
            annot = "authentic"
            
        rows.append({"case_id": Path(f).stem, "annotation": annot})
        
    sub = pd.DataFrame(rows)
    sample = pd.read_csv(CONFIG.BASE_DIR + "/sample_submission.csv")
    sample["case_id"] = sample["case_id"].astype(str)
    final = sample[["case_id"]].merge(sub, on="case_id", how="left").fillna("authentic")
    final.to_csv("submission.csv", index=False)
    print("✅ Submission saved.")

if __name__ == "__main__":
    run_inference()